# 06 — Site map

Renders the five observation points and their eclipse sightline wedges on a local vector basemap.


In [ ]:
# Load the map dependencies and define the raw-data and figure directories.
import json

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds

import matplotlib.pyplot as plt

from eclipse_viewshed.project import repository_root
PROJECT_ROOT = repository_root()

from eclipse_viewshed import aoi, solar, vector

RAW = PROJECT_ROOT / "data" / "raw"
INTERIM = PROJECT_ROOT / "data" / "interim"
GRB = RAW / "grb"
WEGENREGISTER = RAW / "wegenregister"
for directory in (GRB, WEGENREGISTER):
    directory.mkdir(parents=True, exist_ok=True)
FIGDIR = PROJECT_ROOT / "reports" / "figures" / "factsheet"
FIGDIR.mkdir(parents=True, exist_ok=True)

PT, DPI = 1 / 72, 300
INK, TX, TX2, TX3 = "#0E1117", "#E9ECF2", "#949DAF", "#5C6577"
PANEL = "#1C232F"
SUN, WARM = "#EEDD88", "#EE8866"


## Palette


In [ ]:
# Define the map palette, feature colours, and road-width hierarchy.
FIORD = {
    "land":     "#45516E",
    "water":    "#38435C",
    "building": mcolors.hsv_to_rgb((232 / 360, 0.47, 0.18)),
    "road":     mcolors.hsv_to_rgb((224 / 360, 0.22, 0.45)),
}
# Define subdued building and vegetation colours.
BUILDING_HEX = "#0C1222"
VEGETATION_HEX = "#101A38"
DARKEN = 0.55


def toward_ink(colour, amount=DARKEN):
    r, g, b = mcolors.to_rgb(colour)
    ir, ig, ib = mcolors.to_rgb(INK)
    return (r + (ir - r) * amount, g + (ig - g) * amount, b + (ib - b) * amount)


LAND       = toward_ink(FIORD["land"])
# Match the river to the skyline figures' panel colour.
WATER      = PANEL
ROAD       = toward_ink(FIORD["road"], 0.42)
BUILDING   = BUILDING_HEX
VEGETATION = VEGETATION_HEX

# Define map strokes in metres.
MAJOR_M, MINOR_M = 9.0, 4.0
MAJOR_CLASSES = {"autosnelweg", "hoofdweg", "primaire weg", "secundaire weg"}


## Map extent


In [ ]:
# Load the observers and classified raster and calculate the final map extent.
observers = pd.read_csv(PROJECT_ROOT / "data" / "external" / "observers.csv")
observers[["x", "y"]] = observers.apply(
    lambda r: pd.Series(aoi.to_lambert72(r["lat"], r["lon"])), axis=1)

# Use the map extent defined by the acquisition module.
FRAME = aoi.map_frame(list(zip(observers["lat"], observers["lon"])))
west, east, south, north = FRAME.xmin, FRAME.xmax, FRAME.ymin, FRAME.ymax

with rasterio.open(INTERIM / "classes_1m.tif") as src:
    win = from_bounds(west, south, east, north, src.transform)
    classes = src.read(1, window=win)
    b = rasterio.windows.bounds(win, src.transform)

RASTER_EXTENT = [b[0], b[2], b[1], b[3]]
LABEL_GUTTER_M = 350
EXTENT = [b[0], b[2] + LABEL_GUTTER_M, b[1], b[3]]
BOUNDS = aoi.Bounds(b[0], b[1], b[2], b[3])
width_m, height_m = EXTENT[1] - EXTENT[0], EXTENT[3] - EXTENT[2]

FIG_W_PT = 400
FIG_H_PT = FIG_W_PT * height_m / width_m

print(f"window {classes.shape[1]} x {classes.shape[0]} px "
      f"= {width_m:.0f} x {height_m:.0f} m")
print(f"figure {FIG_W_PT} x {FIG_H_PT:.0f} pt")
print(f"bounds {BOUNDS}")


## Roads


In [ ]:
# Load the road segments from the local cache or the published feature service.
ROAD_CACHE = WEGENREGISTER / f"wegenregister_{BOUNDS.xmin:.0f}_{BOUNDS.ymin:.0f}.geojson"
roads_fc = vector.fetch_features(
    vector.LAYER_ROADS,
    BOUNDS,
    out_path=ROAD_CACHE,
    url=vector.WEGENREGISTER_WFS,
    progress=True,
)
print(f"roads: {len(roads_fc['features'])} segments")


## Vector layers


In [ ]:
# Define geometry helpers and load the building and water vector layers.
PAD = 2 * MAJOR_M


def rings(geometry):
    """Every ring of a Polygon or MultiPolygon."""
    if not geometry:
        return
    if geometry["type"] == "Polygon":
        yield from geometry["coordinates"]
    elif geometry["type"] == "MultiPolygon":
        for poly in geometry["coordinates"]:
            yield from poly


def lines(geometry):
    """Every part of a LineString or MultiLineString."""
    if not geometry:
        return
    if geometry["type"] == "LineString":
        yield geometry["coordinates"]
    elif geometry["type"] == "MultiLineString":
        yield from geometry["coordinates"]


def in_window(coords):
    a = np.asarray(coords, dtype=float)
    if a.ndim != 2 or len(a) < 2:
        return None
    if (a[:, 0].max() < EXTENT[0] - PAD or a[:, 0].min() > EXTENT[1] + PAD
            or a[:, 1].max() < EXTENT[2] - PAD or a[:, 1].min() > EXTENT[3] + PAD):
        return None
    return a


# Load the largest cached GRB footprint collection from notebook 03.
cached = sorted(GRB.glob("GRB_GBG_x*.geojson"),
                key=lambda q: q.stat().st_size, reverse=True)
buildings_fc = None

for path in cached:
    parts = path.stem.split("_")
    try:
        cx0, cx1 = int(parts[2][1:]), int(parts[3])
        cy0, cy1 = int(parts[4][1:]), int(parts[5])
    except (IndexError, ValueError):
        continue
    covers = (cx0 <= BOUNDS.xmin and cx1 >= BOUNDS.xmax
              and cy0 <= BOUNDS.ymin and cy1 >= BOUNDS.ymax)
    short_n = max(0, BOUNDS.ymax - cy1)
    short_s = max(0, cy0 - BOUNDS.ymin)
    print(f"  {path.name}")
    print(f"    covers x {cx0}-{cx1}  y {cy0}-{cy1}   "
          f"{'fully covers the window' if covers else 'PARTIAL'}")
    if not covers:
        print(f"    short by {short_n:.0f} m north, {short_s:.0f} m south "
              f"- buildings will be missing there")
    buildings_fc = json.loads(path.read_text(encoding="utf-8"))
    print(f"    {len(buildings_fc['features'])} features loaded from cache")
    break

if buildings_fc is None:
    print("  no GRB cache found; fetching")
    try:
        buildings_fc = vector.fetch_features(
            vector.LAYER_BUILDINGS, BOUNDS,
            out_path=GRB / vector.cache_name(vector.LAYER_BUILDINGS, BOUNDS),
            progress=True)
    except Exception as exc:
        print(f"  NO BUILDINGS: {type(exc).__name__}: {exc}")
        buildings_fc = {"features": []}

building_paths = []
for feat in buildings_fc["features"]:
    for ring in rings(feat.get("geometry")):
        a = in_window(ring)
        if a is None:
            continue
        pts = np.vstack([a, a[:1]])
        codes = ([MplPath.MOVETO] + [MplPath.LINETO] * (len(pts) - 2)
                 + [MplPath.CLOSEPOLY])
        building_paths.append(MplPath(pts, codes))

major, minor = [], []
if roads_fc:
    for feat in roads_fc["features"]:
        cls = str(feat["properties"].get("morfologie")
                  or feat["properties"].get("wegcategorie") or "").lower()
        bucket = major if any(k in cls for k in MAJOR_CLASSES) else minor
        for part in lines(feat.get("geometry")):
            a = in_window(part)
            if a is not None:
                bucket.append(a)

print(f"buildings {len(building_paths)} rings")
print(f"roads     {len(major)} major, {len(minor)} minor")


## Render


In [ ]:
# Render the classified surface, roads, sightline wedges, observers, and labels.
WEDGE_M = 1200
AZ_MAX_ECLIPSE = 284.4

fig, ax = plt.subplots(figsize=(FIG_W_PT * PT, FIG_H_PT * PT), dpi=DPI)
fig.patch.set_facecolor(LAND)
ax.set_facecolor(LAND)

# Draw separate masked building and vegetation layers.
for code, colour, z in ((3, VEGETATION, 1), (2, WATER, 2)):
    layer = np.where(classes == code, 1.0, np.nan)
    ax.imshow(layer, extent=RASTER_EXTENT, origin="upper", interpolation="nearest",
              cmap=mcolors.ListedColormap([colour]), vmin=0, vmax=1, zorder=z)

nodata = float((classes == 0).mean())
if nodata > 0.001:
    print(f"  !! {100 * nodata:.1f}% of the frame has no classified surface - "
          "re-run notebooks 02 and 03 to acquire the map frame")

if building_paths:
    ax.add_collection(PathCollection(building_paths, facecolors=[BUILDING],
                                     edgecolors="none", zorder=3))

m_per_pt = width_m / FIG_W_PT
if minor:
    ax.add_collection(LineCollection(minor, colors=[ROAD], zorder=4,
                                     linewidths=MINOR_M / m_per_pt,
                                     capstyle="round", joinstyle="round"))
if major:
    ax.add_collection(LineCollection(major, colors=[ROAD], zorder=5,
                                     linewidths=MAJOR_M / m_per_pt,
                                     capstyle="round", joinstyle="round"))

# Draw the sightline wedges and observer labels.
for _, row in observers.iterrows():
    ax.add_patch(Wedge((row.x, row.y), WEDGE_M,
                       90 - solar.WEDGE_AZ_MAX, 90 - solar.WEDGE_AZ_MIN,
                       facecolor=SUN, alpha=0.11, edgecolor="none", zorder=6))
    th = np.radians(90 - AZ_MAX_ECLIPSE)
    ax.plot([row.x, row.x + WEDGE_M * np.cos(th)],
            [row.y, row.y + WEDGE_M * np.sin(th)],
            color=SUN, linewidth=0.7, alpha=0.8, zorder=7)

import textwrap

ax.scatter(observers.x, observers.y, s=18, facecolor=WARM,
           edgecolor=INK, linewidth=0.7, zorder=9)

# Wrap long names inside the eastern label gutter.
LABEL_WRAP = 15
for _, row in observers.iterrows():
    ax.annotate(textwrap.fill(row["name"], LABEL_WRAP),
                (row.x, row.y),
                xytext=(7, 5),
                textcoords="offset points",
                ha="left", va="bottom",
                color=TX, fontsize=8.25, linespacing=1.25,
                fontfamily="monospace", zorder=10)

x0 = EXTENT[0] + 0.04 * width_m
y0 = EXTENT[2] + 0.045 * height_m
ax.plot([x0, x0 + 500], [y0, y0], color=TX, linewidth=1.6, zorder=10)
ax.annotate("500 m", (x0 + 250, y0), xytext=(0, 4), textcoords="offset points",
            color=TX, fontsize=7.0, ha="center", fontfamily="monospace",
            zorder=10)

ax.set_xlim(EXTENT[0], EXTENT[1])
ax.set_ylim(EXTENT[2], EXTENT[3])
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig.savefig(FIGDIR / "fig3_site_map.png", dpi=DPI, facecolor=LAND)
print(f"wrote {(FIGDIR / 'fig3_site_map.png').relative_to(PROJECT_ROOT)}")
plt.show()
